# Notebook 10 — Business Rules, Signals & Scoring (Corrected Confidence)

**Input** → `data/indicators.parquet` (with validity status)

**Output** → `data/signals.parquet`

**Key improvement from feedback:**
Confidence Score is now **INDEPENDENT** of Overall Score direction.

| Dimension | Measures | Range | Example |
|---|---|---|---|
| **Overall Score** | Direction & strength of signals | 0–100 | 82 = strongly bullish |
| **Confidence** | Data quality & agreement | 0–100% | 42% = sparse/disagreement |

Valid combinations:
- **Score=82, Confidence=42%**: Strong bullish signal, but low data quality → "bullish with low conviction"
- **Score=48, Confidence=91%**: Weak directional signal, but excellent data → "no strong move, high confidence"

Confidence NEVER looks at whether the score is extreme or neutral — it only evaluates DATA QUALITY.

## Three Confidence Components

| Component | Weight | What it measures |
|---|---|---|
| Data Coverage | 40% | Fraction of indicators with VALID status |
| Family Agreement | 40% | Do Trend/Momentum/Volume families agree on direction? |
| Risk Penalty | 20% | High HV or sparse volume reduces confidence |

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
def _find_root(start):
    for c in [start, start.parent, start.parent.parent]:
        if (c / 'src').exists() and (c / 'data').exists():
            return c.resolve()
    raise RuntimeError(f'Cannot locate project root from {start}')
ROOT = _find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.validation import load_unified_dataset, save_unified_dataset
from config.methodology import (
    SCORE_WEIGHTS, CONFIDENCE_WEIGHTS, MIN_COVERAGE_FOR_DECISION,
    SIGNAL_RULES, WEIGHTS_STATUS,
)
pd.set_option('display.float_format', '{:.4f}'.format)
print(f'ROOT: {ROOT}')
print(f'Weights: {SCORE_WEIGHTS}')
print(f'Weights status: {WEIGHTS_STATUS}  ← BASELINE, not validated yet')

ROOT: /home/yass/Desktop/DSS_CMR
Weights: {'Trend': 0.35, 'Momentum': 0.35, 'Volume': 0.2, 'Risk': 0.1}
Weights status: BASELINE_HYPOTHESIS  ← BASELINE, not validated yet


## Step 1 — Load indicators with validity status

In [2]:
df, _ = load_unified_dataset(str(ROOT / 'data' / 'indicators.parquet'))
print(f'Shape: {df.shape}')
IND_COLS = ['SMA_20','SMA_50','EMA_20','RSI_14','MACD','MACD_Signal','MACD_Histogram','RVOL','VWAP','HV_20']
print(f'\nIndicator coverage:')
for c in IND_COLS:
    valid = (df[f'Valid_{c}'] == 'VALID').sum()
    print(f'  {c:20s}: {valid:4d}/{len(df)} VALID')

Shape: (140, 31)

Indicator coverage:
  SMA_20              :    0/140 VALID
  SMA_50              :    0/140 VALID
  EMA_20              :   70/140 VALID
  RSI_14              :    5/140 VALID
  MACD                :    0/140 VALID
  MACD_Signal         :    0/140 VALID
  MACD_Histogram      :    0/140 VALID
  RVOL                :   30/140 VALID
  VWAP                :   30/140 VALID
  HV_20               :    0/140 VALID


## Step 2 — Individual signals

In [3]:
def individual_signals(row):
    s, c = {}, row['Cours']
    for ind in ['SMA_20','SMA_50','EMA_20']:
        v = row[ind]
        s[ind] = 1 if (pd.notna(v) and pd.notna(c) and c > v) else (-1 if (pd.notna(v) and pd.notna(c) and c < v) else np.nan)
    rsi = row['RSI_14']
    s['RSI_14'] = (1 if (pd.notna(rsi) and rsi < SIGNAL_RULES['RSI_14']['bullish_if_below']) else
                   (-1 if (pd.notna(rsi) and rsi > SIGNAL_RULES['RSI_14']['bearish_if_above']) else 0 if pd.notna(rsi) else np.nan))
    macd, sig = row['MACD'], row['MACD_Signal']
    s['MACD'] = 1 if (pd.notna(macd) and pd.notna(sig) and macd > sig) else (-1 if (pd.notna(macd) and pd.notna(sig)) else np.nan)
    rvol = row['RVOL']
    s['RVOL'] = (1 if (pd.notna(rvol) and rvol >= SIGNAL_RULES['RVOL']['confirm_threshold']) else
                 (-1 if (pd.notna(rvol) and rvol <= SIGNAL_RULES['RVOL']['weak_threshold']) else 0 if pd.notna(rvol) else np.nan))
    vwap = row['VWAP']
    s['VWAP'] = 1 if (pd.notna(vwap) and pd.notna(c) and c > vwap) else (-1 if (pd.notna(vwap) and pd.notna(c)) else np.nan)
    s['HV_20'] = np.nan
    return s

sig_cols = ['SMA_20','SMA_50','EMA_20','RSI_14','MACD','RVOL','VWAP','HV_20']
signals_df = df.apply(individual_signals, axis=1, result_type='expand')
for c in sig_cols:
    df[f'Sig_{c}'] = signals_df[c]
print('Individual signals generated')

Individual signals generated


## Step 3 — Family scores

In [4]:
FAMILIES = {
    'Trend':    ['Sig_SMA_20','Sig_SMA_50','Sig_EMA_20'],
    'Momentum': ['Sig_RSI_14','Sig_MACD'],
    'Volume':   ['Sig_RVOL','Sig_VWAP'],
}
def family_score(row, members):
    vals = [row[m] for m in members if not pd.isna(row.get(m, np.nan))]
    return ((np.mean(vals) + 1) / 2 * 100) if vals else np.nan

for fam, members in FAMILIES.items():
    df[f'Score_{fam}'] = df.apply(lambda r: family_score(r, members), axis=1)
print('Family scores computed')

Family scores computed


## Step 4 — Overall Score

In [5]:
W = SCORE_WEIGHTS
def overall_score(row, weights):
    total_w, ws = 0.0, 0.0
    for fam, w in weights.items():
        if fam == 'Risk': continue
        v = row.get(f'Score_{fam}', np.nan)
        if not pd.isna(v):
            ws += v*w; total_w += w
    return ws/total_w if total_w > 0 else np.nan

df['Overall_Score'] = df.apply(lambda r: overall_score(r, W), axis=1)
print(f'Overall_Score: min={df["Overall_Score"].min():.1f}  max={df["Overall_Score"].max():.1f}')

Overall_Score: min=0.0  max=100.0


## Step 5 — **CORRECTED: Confidence Score (independent of Overall_Score)**

Confidence measures DATA QUALITY, not signal direction.

In [6]:
REQUIRED_IND = ['SMA_20','SMA_50','EMA_20','RSI_14','MACD','RVOL','VWAP']
CW = CONFIDENCE_WEIGHTS

def confidence_score_v2(row, df_all, CW):
    """
    Confidence is INDEPENDENT of Overall_Score.
    Valid: Score=82 + Confidence=42% (strong signal, low data quality).
    
    A. Data Coverage (40%)   — fraction of indicators with VALID status
    B. Family Agreement (40%)— do families agree on direction?
    C. Risk Penalty (20%)    — high HV reduces confidence
    """
    # A. Data coverage from validity status
    valid_count = sum(1 for ind in REQUIRED_IND 
                      if row.get(f'Valid_{ind}', 'INSUFFICIENT_DATA') == 'VALID')
    coverage = valid_count / len(REQUIRED_IND)

    # B. Family agreement (consensus, not direction)
    fam_scores = [row.get(f'Score_{f}', np.nan) for f in ['Trend','Momentum','Volume']]
    valid_fams = [s for s in fam_scores if pd.notna(s)]
    if len(valid_fams) >= 2:
        bullish, bearish = sum(1 for s in valid_fams if s > 50), sum(1 for s in valid_fams if s < 50)
        agreement = max(bullish, bearish) / len(valid_fams)
    elif len(valid_fams) == 1:
        agreement = 0.5
    else:
        agreement = 0.0

    # C. Risk penalty
    hv = row.get('HV_20', np.nan)
    if pd.notna(hv) and len(df_all['HV_20'].dropna()) > 0:
        hv_p75 = df_all['HV_20'].quantile(0.75)
        risk_penalty = min(0.3, max(0.0, (hv - hv_p75)/hv_p75)) if hv_p75 > 0 else 0.0
    else:
        risk_penalty = 0.0

    raw = (coverage * CW['data_coverage'] +
           agreement * CW['family_agreement'] +
           (1 - risk_penalty) * CW['risk_penalty'])
    return round(raw * 100, 1)

df['Confidence'] = df.apply(lambda r: confidence_score_v2(r, df, CW), axis=1)
print(f'Confidence: min={df["Confidence"].min():.1f}  max={df["Confidence"].max():.1f}')
print('\n⚠️  Confidence is now INDEPENDENT of Overall_Score.')
print('   Score=82 + Confidence=42% is a valid and meaningful combination.')

Confidence: min=20.0  max=77.1

⚠️  Confidence is now INDEPENDENT of Overall_Score.
   Score=82 + Confidence=42% is a valid and meaningful combination.


## Step 6 — Summary per company

In [7]:
rows_out = []
for isin, grp in df.groupby('CODE_ISIN'):
    latest = grp.sort_values('Date').dropna(subset=['Overall_Score']).tail(1)
    if len(latest) == 0: latest = grp.sort_values('Date').tail(1)
    r = latest.iloc[0]
    rows_out.append({
        'CODE_ISIN': isin, 'Company': r['Company'], 'Date': r['Date'].date(),
        'Cours': round(r['Cours'],2) if pd.notna(r['Cours']) else np.nan,
        'Overall_Score': round(r['Overall_Score'],1) if pd.notna(r['Overall_Score']) else np.nan,
        'Confidence': r['Confidence'],
        'EMA_Sig': r.get('Sig_EMA_20', np.nan), 'RSI_Sig': r.get('Sig_RSI_14', np.nan),
    })
summary = pd.DataFrame(rows_out)
print('Latest per company:')
print(summary.to_string(index=False))

Latest per company:
   CODE_ISIN            Company       Date     Cours  Overall_Score  Confidence  EMA_Sig  RSI_Sig
MA0000010936 ALUMINIUM DU MAROC 2019-01-21 1600.0000        30.6000     56.2000  -1.0000   0.0000
MA0000010944               AGMA 2019-01-21 3040.0000        50.0000     51.4000  -1.0000   1.0000
MA0000010951       AFRIQUIA GAZ 2019-01-21 3233.0000        75.0000     51.4000   1.0000   0.0000
MA0000011819          ALLIANCES 2019-01-21   77.1000        44.4000     69.5000  -1.0000   1.0000
MA0000012296               AFMA 2019-01-21  960.0000        30.6000     56.2000  -1.0000   0.0000


## Step 7 — Save

In [8]:
rep = save_unified_dataset(df, str(ROOT/'data'/'signals.parquet'))
print(f'✓ data/signals.parquet  {rep["rows"]} rows  {rep["file_size_mb"]:.3f} MB')

✓ data/signals.parquet  140 rows  0.027 MB


## Summary

In [9]:
print('NOTEBOOK 10 — SIGNALS & SCORING'.center(65,'='))
print(f'  Confidence is now INDEPENDENT of Overall_Score.')
print(f'  Score measures DIRECTION, Confidence measures QUALITY.')
print(f'  Weights status: {WEIGHTS_STATUS} ← requires backtesting')
print('='*65)

=================NOTEBOOK 10 — SIGNALS & SCORING=================
  Confidence is now INDEPENDENT of Overall_Score.
  Score measures DIRECTION, Confidence measures QUALITY.
  Weights status: BASELINE_HYPOTHESIS ← requires backtesting
